In [1]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.3
numpy: 2.4.6


In [15]:
pd.set_option('display.width', 150)

In [2]:
# Load the same intermediate artifact used in 03_modeling.ipynb.
train_bureau = pd.read_csv('../data/processed/train_bureau.csv')
print(train_bureau.shape)

(307511, 156)


In [3]:
train_bureau.head(2)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,BUREAU_CREDIT_TYPE_NUNIQUE,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MAX,BUREAU_AMT_ANNUITY_SUM,BUREAU_AMT_ANNUITY_MEAN,BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT,BUREAU_CREDIT_ACTIVE_BAD_DEBT_COUNT,BUREAU_CREDIT_ACTIVE_CLOSED_COUNT,BUREAU_CREDIT_ACTIVE_SOLD_COUNT,BUREAU_CREDIT_TYPE_MODE_COUNT
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,2.0,-499.875,-7.0,0.0,0.0,2.0,0.0,6.0,0.0,4.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,2.0,-816.000,-43.0,0.0,NaN,1.0,0.0,3.0,0.0,2.0


In [7]:
# Quick missing-value audit on train_bureau, freshly loaded in this notebook.
# Purpose: confirm missing counts before choosing which variables to bin first,
# without relying on memory from previous sessions.

missing_summary = pd.DataFrame({
    'dtype': train_bureau.dtypes,
    'n_missing': train_bureau.isnull().sum(),
    'pct_missing': (train_bureau.isnull().sum() / len(train_bureau) * 100).round(2)
})
missing_summary = missing_summary[missing_summary['n_missing'] > 0].sort_values('n_missing')

print("Total de colunas com missing:", len(missing_summary))
print()
print(missing_summary.head(20))

Total de colunas com missing: 98

                                     dtype  n_missing  pct_missing
DAYS_LAST_PHONE_CHANGE             float64          1         0.00
YEARS_LAST_PHONE_CHANGE            float64          1         0.00
CNT_FAM_MEMBERS                    float64          2         0.00
AMT_ANNUITY                        float64         12         0.00
AMT_GOODS_PRICE                    float64        278         0.09
EXT_SOURCE_2                       float64        660         0.21
OBS_60_CNT_SOCIAL_CIRCLE           float64       1021         0.33
OBS_30_CNT_SOCIAL_CIRCLE           float64       1021         0.33
DEF_60_CNT_SOCIAL_CIRCLE           float64       1021         0.33
DEF_30_CNT_SOCIAL_CIRCLE           float64       1021         0.33
NAME_TYPE_SUITE                        str       1292         0.42
AMT_REQ_CREDIT_BUREAU_HOUR         float64      41519        13.50
AMT_REQ_CREDIT_BUREAU_MON          float64      41519        13.50
AMT_REQ_CREDIT_BUREAU_QRT   

In [8]:
# Focused check on the EXT_SOURCE variables specifically, since they're
# the strongest predictors and the natural first candidates for the scorecard.

ext_source_cols = [c for c in train_bureau.columns if c.startswith('EXT_SOURCE')]
print(train_bureau[ext_source_cols].isnull().sum())

EXT_SOURCE_1    173378
EXT_SOURCE_2       660
EXT_SOURCE_3     60965
dtype: int64


In [16]:
# Bin EXT_SOURCE_2 into 10 quantile groups, with an explicit "Missing" bin
# for the 660 null values, instead of letting qcut silently drop them.

woe_df = pd.DataFrame({
    'EXT_SOURCE_2': train_bureau['EXT_SOURCE_2'],
    'TARGET': train_bureau['TARGET']
})

woe_df['bin'] = pd.qcut(woe_df['EXT_SOURCE_2'], q=10, duplicates='drop').astype('object')
woe_df.loc[woe_df['EXT_SOURCE_2'].isnull(), 'bin'] = 'Missing'

print(woe_df['bin'].value_counts(dropna=False))

bin
(0.646, 0.682]            30694
(0.34, 0.44]              30687
(0.566, 0.608]            30687
(-0.0009999183, 0.216]    30686
(0.216, 0.34]             30685
(0.722, 0.855]            30685
(0.512, 0.566]            30684
(0.44, 0.512]             30684
(0.608, 0.646]            30683
(0.682, 0.722]            30676
Missing                     660
Name: count, dtype: int64


In [17]:
# Calculate WoE and IV per bin (10 quantile bins + 1 explicit Missing bin).

total_good = (woe_df['TARGET'] == 0).sum()
total_bad = (woe_df['TARGET'] == 1).sum()

grouped = woe_df.groupby('bin', observed=True).agg(
    n_total=('TARGET', 'count'),
    n_bad=('TARGET', 'sum')
).reset_index()

grouped['n_good'] = grouped['n_total'] - grouped['n_bad']
grouped['pct_good'] = grouped['n_good'] / total_good
grouped['pct_bad'] = grouped['n_bad'] / total_bad
grouped['woe'] = np.log(grouped['pct_good'] / grouped['pct_bad'])
grouped['iv_component'] = (grouped['pct_good'] - grouped['pct_bad']) * grouped['woe']

iv_total = grouped['iv_component'].sum()

print(grouped[['bin', 'n_total', 'n_bad', 'pct_good', 'pct_bad', 'woe', 'iv_component']].round(4))
print()
print("IV total:", round(iv_total, 4))

                       bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
0   (-0.0009999183, 0.216]    30686   5631    0.0886   0.2268 -0.9397        0.1299
1            (0.216, 0.34]    30685   3706    0.0954   0.1493 -0.4474        0.0241
2             (0.34, 0.44]    30687   3056    0.0977   0.1231 -0.2307        0.0058
3            (0.44, 0.512]    30684   2566    0.0995   0.1034 -0.0384        0.0001
4           (0.512, 0.566]    30684   2278    0.1005   0.0918  0.0908        0.0008
5           (0.566, 0.608]    30687   2042    0.1013   0.0823  0.2086        0.0040
6           (0.608, 0.646]    30683   1794    0.1022   0.0723  0.3465        0.0104
7           (0.646, 0.682]    30694   1499    0.1033   0.0604  0.5367        0.0230
8           (0.682, 0.722]    30676   1289    0.1040   0.0519  0.6942        0.0361
9           (0.722, 0.855]    30685    912    0.1053   0.0367  1.0532        0.0722
10                 Missing      660     52    0.0022   0.0021  0.0264       

In [18]:
print(grouped[['woe']].round(4))

       woe
0  -0.9397
1  -0.4474
2  -0.2307
3  -0.0384
4   0.0908
5   0.2086
6   0.3465
7   0.5367
8   0.6942
9   1.0532
10  0.0264


In [13]:
def calculate_woe_iv(df, feature_col, target_col='TARGET', n_bins=10, min_bad_per_bin=100):
    """
    Bin a numeric feature into quantile groups (plus an explicit 'Missing' bin),
    then calculate WoE and IV per bin. Warns if any bin has fewer than
    min_bad_per_bin 'bad' cases, signaling potential WoE instability.

    Returns a DataFrame with bin-level WoE/IV, and the total IV as a float.
    """
    woe_df = pd.DataFrame({
        feature_col: df[feature_col],
        target_col: df[target_col]
    })

    woe_df['bin'] = pd.qcut(woe_df[feature_col], q=n_bins, duplicates='drop').astype('object')
    woe_df.loc[woe_df[feature_col].isnull(), 'bin'] = 'Missing'

    total_good = (woe_df[target_col] == 0).sum()
    total_bad = (woe_df[target_col] == 1).sum()

    grouped = woe_df.groupby('bin', observed=True).agg(
        n_total=(target_col, 'count'),
        n_bad=(target_col, 'sum')
    ).reset_index()

    grouped['n_good'] = grouped['n_total'] - grouped['n_bad']
    grouped['pct_good'] = grouped['n_good'] / total_good
    grouped['pct_bad'] = grouped['n_bad'] / total_bad
    grouped['woe'] = np.log(grouped['pct_good'] / grouped['pct_bad'])
    grouped['iv_component'] = (grouped['pct_good'] - grouped['pct_bad']) * grouped['woe']

    iv_total = grouped['iv_component'].sum()

    unstable_bins = grouped[grouped['n_bad'] < min_bad_per_bin]
    if len(unstable_bins) > 0:
        print(f"AVISO: {len(unstable_bins)} bin(s) com menos de {min_bad_per_bin} casos 'bad', WoE pode ser instável:")
        print(unstable_bins[['bin', 'n_bad']].to_string(index=False))
        print()

    return grouped[['bin', 'n_total', 'n_bad', 'pct_good', 'pct_bad', 'woe', 'iv_component']], iv_total

In [19]:
ext3_woe, ext3_iv = calculate_woe_iv(train_bureau, 'EXT_SOURCE_3')

print(ext3_woe.round(4))
print()
print("IV total (EXT_SOURCE_3):", round(ext3_iv, 4))

                   bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
0   (-0.000473, 0.228]    24701   4941    0.0699   0.1990 -1.0464        0.1351
1        (0.228, 0.33]    24744   3156    0.0764   0.1271 -0.5097        0.0259
2        (0.33, 0.408]    25057   2383    0.0802   0.0960 -0.1796        0.0028
3       (0.408, 0.476]    24689   1970    0.0804   0.0794  0.0127        0.0000
4       (0.476, 0.535]    24186   1494    0.0803   0.0602  0.2881        0.0058
5       (0.535, 0.592]    25392   1357    0.0850   0.0547  0.4417        0.0134
6       (0.592, 0.643]    24725   1173    0.0833   0.0473  0.5672        0.0205
7       (0.643, 0.694]    24745   1043    0.0838   0.0420  0.6910        0.0289
8       (0.694, 0.749]    23675    836    0.0808   0.0337  0.8751        0.0412
9       (0.749, 0.896]    24632    795    0.0843   0.0320  0.9682        0.0506
10             Missing    60965   5677    0.1956   0.2287 -0.1564        0.0052

IV total (EXT_SOURCE_3): 0.3294


In [20]:
def calculate_woe_iv_categorical(df, feature_col, target_col='TARGET', min_bad_per_bin=100):
    """
    Calculate WoE and IV for a categorical feature, using each category
    as its own bin. Missing values (if any) get an explicit 'Missing' category.
    """
    woe_df = pd.DataFrame({
        feature_col: df[feature_col],
        target_col: df[target_col]
    })
    woe_df[feature_col] = woe_df[feature_col].astype('object')
    woe_df[feature_col] = woe_df[feature_col].fillna('Missing')

    total_good = (woe_df[target_col] == 0).sum()
    total_bad = (woe_df[target_col] == 1).sum()

    grouped = woe_df.groupby(feature_col, observed=True).agg(
        n_total=(target_col, 'count'),
        n_bad=(target_col, 'sum')
    ).reset_index().rename(columns={feature_col: 'bin'})

    grouped['n_good'] = grouped['n_total'] - grouped['n_bad']
    grouped['pct_good'] = grouped['n_good'] / total_good
    grouped['pct_bad'] = grouped['n_bad'] / total_bad
    grouped['woe'] = np.log(grouped['pct_good'] / grouped['pct_bad'])
    grouped['iv_component'] = (grouped['pct_good'] - grouped['pct_bad']) * grouped['woe']

    iv_total = grouped['iv_component'].sum()

    unstable_bins = grouped[grouped['n_bad'] < min_bad_per_bin]
    if len(unstable_bins) > 0:
        print(f"AVISO: {len(unstable_bins)} categoria(s) com menos de {min_bad_per_bin} casos 'bad', WoE pode ser instável:")
        print(unstable_bins[['bin', 'n_bad']].to_string(index=False))
        print()

    return grouped.sort_values('woe')[['bin', 'n_total', 'n_bad', 'pct_good', 'pct_bad', 'woe', 'iv_component']], iv_total

In [21]:
educ_woe, educ_iv = calculate_woe_iv_categorical(train_bureau, 'NAME_EDUCATION_TYPE')

print(educ_woe.round(4).to_string(index=False))
print()
print("IV total (NAME_EDUCATION_TYPE):", round(educ_iv, 4))

AVISO: 1 categoria(s) com menos de 100 casos 'bad', WoE pode ser instável:
            bin  n_bad
Academic degree      3

                          bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
              Lower secondary     3816    417    0.0120   0.0168 -0.3343        0.0016
Secondary / secondary special   218391  19524    0.7035   0.7865 -0.1115        0.0093
            Incomplete higher    10277    872    0.0333   0.0351 -0.0543        0.0001
             Higher education    74863   4009    0.2506   0.1615  0.4396        0.0392
              Academic degree      164      3    0.0006   0.0001  1.5503        0.0007

IV total (NAME_EDUCATION_TYPE): 0.0508


In [22]:
# Merge 'Academic degree' (only 164 clients, unstable WoE) into 'Higher education',
# the closest category in both WoE and business meaning (both represent
# post-secondary education).

train_bureau['NAME_EDUCATION_TYPE_BINNED'] = train_bureau['NAME_EDUCATION_TYPE'].replace(
    {'Academic degree': 'Higher education'}
)

print(train_bureau['NAME_EDUCATION_TYPE_BINNED'].value_counts())

NAME_EDUCATION_TYPE_BINNED
Secondary / secondary special    218391
Higher education                  75027
Incomplete higher                 10277
Lower secondary                    3816
Name: count, dtype: int64


C:\Users\vitor\AppData\Local\Temp\ipykernel_39576\3584352028.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_bureau['NAME_EDUCATION_TYPE_BINNED'] = train_bureau['NAME_EDUCATION_TYPE'].replace(


In [23]:
educ_woe_v2, educ_iv_v2 = calculate_woe_iv_categorical(train_bureau, 'NAME_EDUCATION_TYPE_BINNED')

print(educ_woe_v2.round(4).to_string(index=False))
print()
print("IV total (NAME_EDUCATION_TYPE_BINNED):", round(educ_iv_v2, 4))
print("Comparação, IV original (5 categorias):", round(educ_iv, 4))

                          bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
              Lower secondary     3816    417    0.0120   0.0168 -0.3343        0.0016
Secondary / secondary special   218391  19524    0.7035   0.7865 -0.1115        0.0093
            Incomplete higher    10277    872    0.0333   0.0351 -0.0543        0.0001
             Higher education    75027   4012    0.2512   0.1616  0.4411        0.0395

IV total (NAME_EDUCATION_TYPE_BINNED): 0.0505
Comparação, IV original (5 categorias): 0.0508
